In [1]:
!pip install -q openai-whisper
!pip install -q librosa
!pip install -q soundfile
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 2.8 MB/s eta 0:00:00


In [2]:
import whisper
import re
import gradio as gr

from collections import Counter

In [3]:
print("Loading Whisper model...")

whisper_model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 124MiB/s]


Whisper model loaded successfully!


In [4]:
def speech_to_text(audio_file):

    result = whisper_model.transcribe(
        audio_file
    )

    transcript = result["text"].strip()

    return transcript

In [5]:
complaint_categories = {

    "Technical Issue": [
        "computer",
        "laptop",
        "software",
        "system",
        "internet",
        "network",
        "website",
        "login",
        "app",
        "application",
        "server"
    ],

    "Product Issue": [
        "product",
        "damaged",
        "broken",
        "defective",
        "quality",
        "item",
        "wrong product"
    ],

    "Service Issue": [
        "service",
        "support",
        "staff",
        "employee",
        "response",
        "delay"
    ],

    "Payment Issue": [
        "payment",
        "money",
        "refund",
        "transaction",
        "amount",
        "billing",
        "bill"
    ],

    "Delivery Issue": [
        "delivery",
        "delivered",
        "package",
        "parcel",
        "order",
        "shipping",
        "late"
    ]
}

In [6]:
def detect_category(text):

    text_lower = text.lower()

    category_scores = {}

    for category, keywords in complaint_categories.items():

        score = 0

        for keyword in keywords:

            if keyword in text_lower:
                score += 1

        category_scores[category] = score

    best_category = max(
        category_scores,
        key=category_scores.get
    )

    if category_scores[best_category] == 0:
        best_category = "General Complaint"

    return best_category

In [7]:
priority_keywords = {

    "High": [
        "urgent",
        "emergency",
        "critical",
        "immediately",
        "danger",
        "serious",
        "not working"
    ],

    "Medium": [
        "delay",
        "problem",
        "issue",
        "late",
        "error"
    ],

    "Low": [
        "request",
        "question",
        "information",
        "suggestion"
    ]
}


def detect_priority(text):

    text_lower = text.lower()

    # Check High priority first
    for keyword in priority_keywords["High"]:
        if keyword in text_lower:
            return "High"

    # Check Medium priority
    for keyword in priority_keywords["Medium"]:
        if keyword in text_lower:
            return "Medium"

    # Check Low priority
    for keyword in priority_keywords["Low"]:
        if keyword in text_lower:
            return "Low"

    # Default priority
    return "Low"

In [8]:
def extract_keywords(text):

    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())

    common_words = {
        "this", "that", "with", "have", "from",
        "there", "about", "please", "very",
        "been", "they", "their", "what",
        "when", "where", "which", "because"
    }

    filtered_words = [
        word for word in words
        if word not in common_words
    ]

    keyword_counts = Counter(filtered_words)

    return keyword_counts.most_common(5)

In [9]:
def analyze_complaint(audio_file):

    if audio_file is None:
        return (
            "Please record or upload a complaint.",
            "",
            "",
            "",
            ""
        )

    # Convert speech to text
    transcript = speech_to_text(audio_file)

    if not transcript:
        return (
            "Could not understand the complaint.",
            "",
            "",
            "",
            ""
        )

    # Detect category
    category = detect_category(transcript)

    # Detect priority
    priority = detect_priority(transcript)

    # Extract keywords
    keywords = extract_keywords(transcript)

    if keywords:
        keyword_text = ", ".join(
            f"{word} ({count})"
            for word, count in keywords
        )
    else:
        keyword_text = "No keywords detected"

    # Create complaint report
    report = (
        f"Complaint Status: Registered\n"
        f"Category: {category}\n"
        f"Priority: {priority}\n"
        f"Important Keywords: {keyword_text}"
    )

    return (
        transcript,
        category,
        priority,
        keyword_text,
        report
    )

In [10]:
with gr.Blocks(title="Voice-Based Complaint Management System") as app:

    gr.Markdown("""
    # 📢 Voice-Based Complaint Management System

    ### 🎤 Record or upload a complaint and automatically analyze it

    **Speech → Whisper ASR → Complaint Text → Category → Priority → Keywords → Report**
    """)

    audio_input = gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="🎤 Record / Upload Complaint"
    )

    analyze_button = gr.Button(
        "🔍 Analyze Complaint",
        variant="primary"
    )

    transcript_output = gr.Textbox(
        label="📝 Complaint Text",
        lines=6
    )

    category_output = gr.Textbox(
        label="📂 Complaint Category"
    )

    priority_output = gr.Textbox(
        label="🚨 Priority"
    )

    keywords_output = gr.Textbox(
        label="🔑 Important Keywords"
    )

    report_output = gr.Textbox(
        label="📋 Complaint Report",
        lines=6
    )

    analyze_button.click(
        fn=analyze_complaint,
        inputs=audio_input,
        outputs=[
            transcript_output,
            category_output,
            priority_output,
            keywords_output,
            report_output
        ]
    )


app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://474728e81860e3b734.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
